# L10. Image Segmentation with YOLO

YOLO segmentation 모델로 객체의 bounding box와 mask를 함께 확인합니다.


## 1. Segmentation이란?

segmentation은 객체의 영역 전체를 픽셀 단위로 예측하는 문제입니다. 출력에는 class, confidence, mask, bounding box가 포함됩니다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'


## 2. 준비

이 실습도 `ultralytics` 패키지를 사용합니다. 필요한 경우 설치 셀의 주석을 해제하고 한 번만 실행하세요.


In [ ]:
# 필요한 경우 한 번만 실행하세요.
# %pip install ultralytics requests pillow matplotlib


In [ ]:
import requests
from ultralytics import YOLO

sample_dir = Path('data/yolo_seg_samples')
sample_dir.mkdir(parents=True, exist_ok=True)

image_urls = {
    'bus.jpg': 'https://ultralytics.com/images/bus.jpg',
    'zidane.jpg': 'https://ultralytics.com/images/zidane.jpg',
}

for filename, url in image_urls.items():
    path = sample_dir / filename
    if not path.exists():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        path.write_bytes(response.content)
        print('downloaded', path)
    else:
        print('exists', path)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, filename in zip(axes, image_urls.keys()):
    image = Image.open(sample_dir / filename)
    ax.imshow(image)
    ax.set_title(filename)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 3. YOLO Segmentation 추론

`yolov8n-seg.pt` 모델을 사용합니다.


In [ ]:
model = YOLO('yolov8n-seg.pt')
results = model(
    [str(sample_dir / 'bus.jpg'), str(sample_dir / 'zidane.jpg')],
    verbose=False,
)

print('number of result objects:', len(results))


In [ ]:
for result in results:
    rendered = result.plot()
    plt.figure(figsize=(6, 4))
    plt.imshow(rendered[..., ::-1])
    plt.title(Path(result.path).name)
    plt.axis('off')
    plt.show()


## 4. 결과 읽기

각 객체에 대해 class, confidence, box, mask 존재 여부를 확인합니다.


In [ ]:
for result in results:
    print('\nimage:', Path(result.path).name)
    boxes = result.boxes
    masks = result.masks
    num_instances = 0 if boxes is None else len(boxes)
    print('instances:', num_instances)

    if boxes is None:
        continue

    for i in range(num_instances):
        cls_id = int(boxes.cls[i])
        conf = float(boxes.conf[i])
        cls_name = result.names[cls_id]
        xyxy = boxes.xyxy[i].tolist()
        has_mask = masks is not None and masks.data is not None and i < len(masks.data)
        rounded_box = list(map(lambda v: round(v, 1), xyxy))
        print(f' class={cls_name}, conf={conf:.3f}, box={rounded_box}, has_mask={has_mask}')


## 5. Mask 시각화

첫 번째 이미지에 대해 mask를 직접 덧씌워 봅니다.


In [ ]:
result = results[0]
image = np.array(Image.open(result.path).convert('RGB'))
overlay = image.copy().astype(float)

colors = [
    np.array([255, 0, 0]),
    np.array([0, 255, 0]),
    np.array([0, 0, 255]),
    np.array([255, 255, 0]),
    np.array([255, 0, 255]),
]

if result.masks is not None:
    masks = result.masks.data.cpu().numpy()

    for i, mask in enumerate(masks):
        color = colors[i % len(colors)]
        mask_bool = mask > 0.5

        if mask_bool.shape != overlay.shape[:2]:
            mask_uint8 = (mask_bool.astype(np.uint8) * 255)
            mask_uint8 = np.array(
                Image.fromarray(mask_uint8).resize(
                    (overlay.shape[1], overlay.shape[0]),
                    resample=Image.Resampling.NEAREST,
                )
            )
            mask_bool = mask_uint8 > 0

        overlay[mask_bool] = 0.6 * overlay[mask_bool] + 0.4 * color

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(overlay.astype(np.uint8))

if result.boxes is not None:
    for i in range(len(result.boxes)):
        x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
        conf = float(result.boxes.conf[i])
        cls_id = int(result.boxes.cls[i])
        cls_name = result.names[cls_id]

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor='white',
            facecolor='none',
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            max(y1 - 5, 5),
            f'{cls_name} {conf:.2f}',
            color='white',
            fontsize=9,
            bbox=dict(facecolor='black', alpha=0.5, pad=2),
        )

ax.set_title('Segmentation Masks + Bounding Boxes')
ax.axis('off')
plt.tight_layout()
plt.show()


## 6. Fine-tuning 예시

segmentation 모델도 작은 공개 데이터셋으로 fine-tuning할 수 있습니다.


In [ ]:
# 실행 시간이 걸릴 수 있으므로 필요할 때만 실행하세요.
#
# train_model = YOLO('yolov8n-seg.pt')
# train_model.train(data='coco8-seg.yaml', epochs=1, imgsz=640, batch=8)


## 7. 핵심 정리

- segmentation은 객체 영역 전체를 예측하는 문제입니다.
- YOLO segmentation 모델은 box와 mask를 함께 제공할 수 있습니다.
- pretrained 모델로도 segmentation을 바로 확인할 수 있습니다.
- mask를 시각화하면 detection보다 더 세밀한 위치 정보를 볼 수 있습니다.

## 8. 연습 문제

1. 두 이미지에서 segmentation mask가 표시된 객체를 정리해보세요.
2. confidence threshold를 바꾸면 mask 결과가 어떻게 달라지는지 확인해보세요.
3. 다른 공개 이미지를 하나 더 다운로드해 segmentation 모델을 실행해보세요.
4. `yolov8n-seg.pt` 대신 더 큰 segmentation 모델을 사용해보세요.
5. segmentation 결과를 detection 결과와 비교해서 어떤 점이 더 자세한지 설명해보세요.


## 9. 연습문제 풀이

아래 셀들은 앞에서 만든 segmentation `model`, `results`, `sample_dir`, `image_urls`를 그대로 사용합니다.


### Exercise 1 풀이

segmentation mask가 표시된 객체를 이미지별로 정리합니다. mask가 있는 객체만 `has_mask=True`로 표시됩니다.


In [ ]:
def summarize_segments(results):
    rows = []
    for result in results:
        boxes = result.boxes
        masks = result.masks
        if boxes is None:
            continue
        for i in range(len(boxes)):
            cls_id = int(boxes.cls[i])
            rows.append({
                'image': Path(result.path).name,
                'class': result.names[cls_id],
                'confidence': float(boxes.conf[i]),
                'has_mask': masks is not None and masks.data is not None and i < len(masks.data),
                'box': [round(v, 1) for v in boxes.xyxy[i].tolist()],
            })
    return rows

segment_rows = summarize_segments(results)
for row in segment_rows:
    print(
        f"{row['image']:10s} | {row['class']:12s} | "
        f"conf={row['confidence']:.3f} | has_mask={row['has_mask']} | box={row['box']}"
    )


### Exercise 2 풀이

confidence threshold를 높이면 낮은 확신도의 instance와 그 mask가 제거되어 mask 개수가 줄어드는 경향이 있습니다.


In [ ]:
def count_segments(result):
    boxes = result.boxes
    masks = result.masks
    box_count = 0 if boxes is None else len(boxes)
    mask_count = 0 if masks is None or masks.data is None else len(masks.data)
    return box_count, mask_count

for conf in [0.25, 0.50, 0.70]:
    threshold_result = model(str(sample_dir / 'bus.jpg'), conf=conf, verbose=False)[0]
    box_count, mask_count = count_segments(threshold_result)
    print(f'conf={conf:.2f} -> boxes={box_count}, masks={mask_count}')


### Exercise 3 풀이

다른 공개 이미지로 segmentation 모델을 실행합니다. 아래 예시는 Darknet 예제 이미지인 `dog.jpg`를 사용합니다.


In [ ]:
extra_url = 'https://raw.githubusercontent.com/pjreddie/darknet/master/data/dog.jpg'
extra_path = sample_dir / 'dog.jpg'

if not extra_path.exists():
    response = requests.get(extra_url, timeout=30)
    response.raise_for_status()
    extra_path.write_bytes(response.content)
    print('downloaded', extra_path)
else:
    print('exists', extra_path)

extra_results = model(str(extra_path), verbose=False)
extra_rendered = extra_results[0].plot()

plt.figure(figsize=(6, 4))
plt.imshow(extra_rendered[..., ::-1])
plt.title('dog.jpg segmentation')
plt.axis('off')
plt.show()

for row in summarize_segments(extra_results):
    print(f"{row['class']:12s} | conf={row['confidence']:.3f} | has_mask={row['has_mask']} | box={row['box']}")


### Exercise 4 풀이

`yolov8s-seg.pt`는 `yolov8n-seg.pt`보다 큰 segmentation 모델입니다. 보통 더 느리지만 더 안정적인 mask를 기대할 수 있습니다.


In [ ]:
small_seg_result = model(str(sample_dir / 'bus.jpg'), verbose=False)[0]
large_seg_model = YOLO('yolov8s-seg.pt')
large_seg_result = large_seg_model(str(sample_dir / 'bus.jpg'), verbose=False)[0]

print('yolov8n-seg boxes/masks:', count_segments(small_seg_result))
print('yolov8s-seg boxes/masks:', count_segments(large_seg_result))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(small_seg_result.plot()[..., ::-1])
axes[0].set_title('yolov8n-seg.pt')
axes[0].axis('off')
axes[1].imshow(large_seg_result.plot()[..., ::-1])
axes[1].set_title('yolov8s-seg.pt')
axes[1].axis('off')
plt.tight_layout()
plt.show()


### Exercise 5 풀이

Detection은 객체를 사각형 bounding box로 표현합니다. Segmentation은 객체가 차지하는 픽셀 영역을 mask로 표현하므로 물체의 실제 윤곽, 겹침, 배경과의 경계를 더 자세히 볼 수 있습니다.


In [ ]:
detection_model = YOLO('yolov8n.pt')
detection_result = detection_model(str(sample_dir / 'bus.jpg'), verbose=False)[0]
segmentation_result = model(str(sample_dir / 'bus.jpg'), verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(detection_result.plot()[..., ::-1])
axes[0].set_title('Detection: boxes')
axes[0].axis('off')
axes[1].imshow(segmentation_result.plot()[..., ::-1])
axes[1].set_title('Segmentation: masks + boxes')
axes[1].axis('off')
plt.tight_layout()
plt.show()
